In [ ]:
# Cek CUDA dan GPU
import torch
import tensorflow as tf

print("=" * 40)
print("GPU STATUS CHECK")
print("=" * 40)

# PyTorch
print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

# TensorFlow
print(f"\nTensorFlow GPU: {tf.config.list_physical_devices('GPU')}")

# System info
print(f"\nGPU Detail:")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

GPU STATUS CHECK

PyTorch CUDA: True
GPU Name: Tesla T4
CUDA Version: 12.8

TensorFlow GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

GPU Detail:
Tesla T4, 15360 MiB, 580.82.07


#Ambil Dataset dari Kaggle

In [ ]:
# Upolad ke Google Colab

from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nalaprogroup","key":"6e69ae1b1b69e1a4d23d6edf0680d84f"}'}

In [ ]:
# Set Permission
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Install Kaggle
!pip install kaggle

In [ ]:
# Download Dataset
# url : https://www.kaggle.com/datasets/nalaprogroup/clean-data-project2-v2
!kaggle datasets download -d nalaprogroup/clean-data-project2-v2

Dataset URL: https://www.kaggle.com/datasets/nalaprogroup/clean-data-project2-v2
License(s): CC0-1.0
100% 3.82M/3.82M [00:00<00:00, 195MB/s]



In [ ]:
# Extract File
!unzip clean-data-project2-v2.zip

Archive:  clean-data-project2-v2.zip
  inflating: dev_text_clean.json     
  inflating: test_text_clean.json    
  inflating: train_text_clean.json   


In [ ]:
# Memindahkan file ke sample_data/Liputan6
!mkdir -p /content/sample_data/Liputan6

!mv dev_data_convert.json /content/sample_data/Liputan6
!mv test_data_convert.json /content/sample_data/Liputan6
!mv train_data_convert.json /content/sample_data/Liputan6

!mv dev_text_clean.json /content/sample_data/Liputan6
!mv test_text_clean.json /content/sample_data/Liputan6
!mv train_text_clean.json /content/sample_data/Liputan6

mv: cannot stat 'dev_data_convert.json': No such file or directory
mv: cannot stat 'test_data_convert.json': No such file or directory
mv: cannot stat 'train_data_convert.json': No such file or directory


In [ ]:
!pip install -i https://pypi.tuna.tsinghua.edu.cn/simple evaluate

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 203.3 kB/s eta 0:00:00


In [ ]:
# fine_tune_t5_summarization.py

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from evaluate import load
import numpy as np
import os


In [ ]:
# KONFIGURASI

MODEL_NAME = "t5-base"  # atau "google/mt5-base" untuk multilingual
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 4  # Sesuaikan dengan GPU (4 untuk 8GB VRAM)
LEARNING_RATE = 3e-4  # Untuk T5, LR lebih besar dari BERT
EPOCHS = 5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "./t5-summarization-indo"

In [ ]:
# LOAD DATA

def load_data(json_path):
    df = pd.read_json(json_path)
    return Dataset.from_pandas(df)

# Load training dan validation data
train_dataset = load_data("/content/sample_data/Liputan6/train_text_clean.json")
val_dataset = load_data("/content/sample_data/Liputan6/dev_text_clean.json")

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Sample article: {train_dataset[0]['clean_article']}...")
print(f"Sample summary: {train_dataset[0]['clean_summary']}")

Training samples: 5000
Validation samples: 1071
Sample article: Puluhan petani di Polewali Mandar, Sulawesi Barat, membakar lahan pertanian mereka. Ini dilakukan sebagai bentuk protes atas kinerja pemerintah yang dinilai lamban menangani bencana kekeringan akibat jebolnya bendungan Sekka-Sekka pasacabanjir bandang Januari lalu. Jebolnya bendungan membuat distribusi air terhenti. Menurut Abdul Rasyid, petani setempat, mereka kesal dan frustrasi karena padinya tak segera mendapat pasokan air yang cukup. Bahkan, sebagian petani menyerahkan lahannya pada peternak sapi untuk dijadikan pakan ternak. Kepala Dusun Nepo, Kecamatan Wonomulyo, Logawali, menjelaskan, bendungan sudah hampir dua bulan rusak. Namun hingga kini belum ada upaya perbaikan yang berarti. Pemerintah beralasan karena tidak ada dana perbaikan. Bencana kekeringan akibat kurangnya pasokan air merupakan yang kesekian kalinya. Padahal setiap kali musim tanam para petani harus mengeluarkan biaya hingga jutaan rupiah. Bahkan tidak

In [ ]:
# LOAD TOKENIZER DAN MODEL

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Tambahkan pad token jika belum ada
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:

# PREPROCESSING (TOKENIZATION)

# Prefix untuk T5 (wajib!)
PREFIX = "summarize: "

def preprocess_function(examples):
    # Input dengan prefix "summarize: "
    inputs = [PREFIX + doc for doc in examples["clean_article"]]

    # Tokenisasi input
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    # Tokenisasi target (summary)
    labels = tokenizer(
        examples["clean_summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Terapkan preprocessing
print("Tokenizing training data...")
tokenized_train = train_dataset.map(preprocess_function, batched=True)
print("Tokenizing validation data...")
tokenized_val = val_dataset.map(preprocess_function, batched=True)

Tokenizing training data...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing validation data...


Map:   0%|          | 0/1071 [00:00<?, ? examples/s]

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0110c86169efbd40e3e900afa5470dcbc542519b2f8609c92888fbb43fc3fcc9
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# METRIK EVALUASI (ROUGE)

rouge = load("rouge")

import numpy as np
from rouge_score import rouge_scorer

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Perbaikan 1: Cek bentuk predictions
    # Jika predictions adalah tuple, ambil yang pertama
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Perbaikan 2: Jika predictions masih logits (3 dimensi), ambil argmax
    if len(predictions.shape) == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Perbaikan 3: Decode satu per satu (hindari batch_decode)
    decoded_preds = []
    for pred in predictions:
        # Filter padding tokens
        pred_ids = [p for p in pred if p != -100 and p != tokenizer.pad_token_id]
        # Decode sebagai list of integers
        decoded = tokenizer.decode(pred_ids, skip_special_tokens=True)
        decoded_preds.append(decoded)

    # Perbaikan 4: Sama untuk labels
    decoded_labels = []
    for label in labels:
        label_ids = [l for l in label if l != -100 and l != tokenizer.pad_token_id]
        decoded = tokenizer.decode(label_ids, skip_special_tokens=True)
        decoded_labels.append(decoded)

    # Hitung ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1 = []
    rouge2 = []
    rougeL = []

    for pred, label in zip(decoded_preds, decoded_labels):
        if pred and label:  # Pastikan tidak kosong
            scores = scorer.score(pred, label)
            rouge1.append(scores['rouge1'].fmeasure)
            rouge2.append(scores['rouge2'].fmeasure)
            rougeL.append(scores['rougeL'].fmeasure)

    return {
        'eval_rouge1': np.mean(rouge1) if rouge1 else 0.0,
        'eval_rouge2': np.mean(rouge2) if rouge2 else 0.0,
        'eval_rougeL': np.mean(rougeL) if rougeL else 0.0,
    }

In [ ]:
# DATA COLLATOR (untuk dynamic padding)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)

In [ ]:
# TRAINING ARGUMENTS

import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch", # dari epoch diganti sementara no
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),  # Aktifkan jika ada GPU
    push_to_hub=False,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=False, #sementara
    metric_for_best_model="eval_loss",
    greater_is_better=True,
    warmup_steps=WARMUP_STEPS,
    report_to="none",  # Matikan wandb/tensorboard jika tidak perlu
)

In [ ]:
# TRAINER

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# MULAI TRAINING

print("\n" + "="*50)
print("STARTING FINE-TUNING...")
print("="*50 + "\n")

trainer.train()


STARTING FINE-TUNING...



Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,0.682540,0.867950,0.356353,0.180403,0.285296
2,0.656465,0.842135,0.343197,0.171508,0.277349
3,1.468015,1.185028,0.270586,0.126568,0.229431
4,1.443839,1.185194,0.270351,0.126890,0.229584
5,1.375225,1.185183,0.269482,0.126921,0.228617


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,0.865714,0.934232,0.349612,0.175976,0.280187


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=1.0387039166259766, metrics={'train_runtime': 7624.7741, 'train_samples_per_second': 3.279, 'train_steps_per_second': 0.82, 'total_flos': 1.5223947264e+16, 'train_loss': 1.0387039166259766, 'epoch': 5.0})

In [ ]:
# EVALUASI FINAL

print("\n" + "="*50)
print("FINAL EVALUATION ON VALIDATION SET")
print("="*50)

eval_results = trainer.evaluate()
print(f"\nROUGE-1: {eval_results['eval_rouge1']:.4f}")
print(f"ROUGE-2: {eval_results['eval_rouge2']:.4f}")
print(f"ROUGE-L: {eval_results['eval_rougeL']:.4f}")


FINAL EVALUATION ON VALIDATION SET



ROUGE-1: 0.2695
ROUGE-2: 0.1269
ROUGE-L: 0.2286


In [ ]:
# VISUALISASI HASIL EVALUASI

import matplotlib.pyplot as plt

# Data skor ROUGE
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
scores = [{eval_results['eval_rouge1']:.4f}, {eval_results['eval_rouge2']:.4f}, {eval_results['eval_rougeL']:.4f}]

# Membuat bar chart
plt.figure(figsize=(6, 4))
bars = plt.bar(metrics, scores)

# Menambahkan label nilai di atas batang
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f'{yval:.4f}', ha='center', va='bottom')

# Judul dan label sumbu
plt.title('ROUGE Scores Evaluation', fontsize=14)
plt.ylabel('Score', fontsize=12)
plt.xlabel('Metrics', fontsize=12)
plt.ylim(0, 0.5)  # Skala 0-0.5 karena skor maksimal 1
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Tampilkan plot
plt.tight_layout()
plt.show()

In [ ]:
# SIMPAN MODEL

print("\nSaving model...")
model.save_pretrained(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")

print(f"\n✅ Model saved to {OUTPUT_DIR}/final_model")